# Sequence File Generation

This notebook converts the wide-format time series matrices into sliding-window (X, y) pairs
suitable for LSTM training. It handles two model variants:

1. **Univariate** — 12 months of fishing hours → predict next month's fishing hours
2. **Multivariate** — 12 months of (fishing hours, boat pings, sightings) → predict next month's fishing hours

The results are written to Parquet files that the training scripts consume directly.

### Univariate Model Data Sequency

In [ ]:
import pandas as pd
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view
import os
import pyarrow.parquet as pq
import pyarrow as pa
from tqdm import tqdm

input_file = '/mnt/shared_data/finflow/train_val_test_splits/train_total_fishing_hours.parquet'
output_dir = '/mnt/shared_data/finflow/precomputed_sequences'
os.makedirs(output_dir, exist_ok=True)

lookback = 12
chunk_size = 500000 # Process 500k hex cells at a time to keep RAM safe

print("Starting background pre-computation...")

# Read the Parquet file in chunks
parquet_file = pq.ParquetFile(input_file)
chunk_iterator = parquet_file.iter_batches(batch_size=chunk_size)

for i, batch in enumerate(tqdm(chunk_iterator, desc="Processing Chunks")):
    # Convert chunk to Pandas
    df = batch.to_pandas()
    
    # Drop index columns to isolate the time-series math
    cols = [c for c in df.columns if c not in ['h3_index', 'index']]
    data = df[cols].values.astype(np.float32)
    
    # 1. THE NUMPY MAGIC (C-level sliding window)
    windows = sliding_window_view(data, window_shape=lookback + 1, axis=1)
    
    # 2. EXPLODE THE DATA (Flatten into flashcards)
    windows = windows.reshape(-1, lookback + 1)
    
    # 3. SPLIT INTO X and y
    X = windows[:, :-1]
    y = windows[:, -1]
    
    # 4. SAVE TO A NEW PARQUET FILE
    # Convert numpy arrays to lists so Parquet can store the X array in a single column
    chunk_df = pd.DataFrame({
        'X': list(X), 
        'y': y
    })
    
    chunk_df.to_parquet(f'{output_dir}/train_sequences_chunk_{i}.parquet', index=False)

print(f"\nDone! Pre-computed sequences saved to {output_dir}")

### Multivariate Model Data Sequency

In [ ]:
import pyarrow.parquet as pq
import pandas as pd
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view
import os
from tqdm import tqdm

def process_aligned_parquets(split_name, input_dir, output_dir, lookback=12, batch_size=15000):
    print(f"\n--- Processing {split_name.upper()} Set ---")
    os.makedirs(output_dir, exist_ok=True)
    
    f_path = f"{input_dir}/{split_name}_total_fishing_hours.parquet"
    p_path = f"{input_dir}/{split_name}_boat_ping_count.parquet"
    s_path = f"{input_dir}/{split_name}_total_sighting_count.parquet"
    
    # 1. Open the Parquet files for streaming (does not load data into RAM)
    pf_f = pq.ParquetFile(f_path)
    pf_p = pq.ParquetFile(p_path)
    pf_s = pq.ParquetFile(s_path)
    
    # 2. Safety Check: Ensure they are perfectly aligned
    assert pf_f.metadata.num_rows == pf_p.metadata.num_rows == pf_s.metadata.num_rows, "CRITICAL ERROR: Matrices have different row counts. They were corrupted during the crash."
    
    total_rows = pf_f.metadata.num_rows
    print(f"Verified! Streaming {total_rows:,} perfectly aligned rows...")
    
    # 3. Create synchronized iterators
    iter_f = pf_f.iter_batches(batch_size=batch_size)
    iter_p = pf_p.iter_batches(batch_size=batch_size)
    iter_s = pf_s.iter_batches(batch_size=batch_size)
    
    # Calculate total batches for the progress bar
    total_batches = (total_rows // batch_size) + 1
    
    # 4. Stream and process simultaneously
    for batch_idx, (b_f, b_p, b_s) in enumerate(tqdm(zip(iter_f, iter_p, iter_s), total=total_batches)):
        
        # Convert just this tiny slice to pandas
        df_f = b_f.to_pandas()
        df_p = b_p.to_pandas()
        df_s = b_s.to_pandas()
        
        time_cols = [c for c in df_f.columns if c != 'h3_index']
        
        # Extract as float32 to save RAM
        arr_f = df_f[time_cols].values.astype(np.float32)
        arr_p = df_p[time_cols].values.astype(np.float32)
        arr_s = df_s[time_cols].values.astype(np.float32)
        
        # Stack into 3D array
        data_3d = np.stack([arr_f, arr_p, arr_s], axis=2)
        
        # C-level sliding window
        windows = sliding_window_view(data_3d, window_shape=lookback + 1, axis=1)
        windows = np.moveaxis(windows, 3, 2)
        windows_flat = windows.reshape(-1, lookback + 1, 3)
        
        # Split X and y
        X = windows_flat[:, :-1, :]
        y = windows_flat[:, -1, 0] # Fishing is index 0
        
        # Save chunk
        chunk_df = pd.DataFrame({'X': X.reshape(X.shape[0], -1).tolist(), 'y': y})
        chunk_df.to_parquet(f'{output_dir}/{split_name}_sequences_multi_chunk_{batch_idx}.parquet', index=False)

INPUT_DIR = '/mnt/shared_data/finflow/train_val_test_splits'
TRAIN_OUT = '/mnt/shared_data/finflow/precomputed_sequences_multi_train'
VAL_OUT = '/mnt/shared_data/finflow/precomputed_sequences_multi_val'

process_aligned_parquets('val', INPUT_DIR, VAL_OUT)
process_aligned_parquets('train', INPUT_DIR, TRAIN_OUT)

print("\n All data is processed completely safely!")